# Colab Training Workflow

Use this notebook to mount Drive, find both datasets (RDD_SPLIT + pothole), convert them into a combined ImageFolder layout, train the baseline classifier, and save artifacts for reuse.

Preferred dataset locations in Colab:
- /content/drive/MyDrive/RDD_SPLIT
- /content/drive/MyDrive/pothole

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 3: Locate both datasets from exact Google Drive paths
from pathlib import Path

rdd_root = Path('/content/drive/MyDrive/RDD_SPLIT')
pothole_root = Path('/content/drive/MyDrive/pothole')

def is_rdd_split_root(path: Path):
    return (
        path.is_dir()
        and (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'labels').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'labels').is_dir()
    )

def is_pothole_seg_root(path: Path):
    return (
        path.is_dir()
        and (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'labels').is_dir()
        and (path / 'valid' / 'images').is_dir()
        and (path / 'valid' / 'labels').is_dir()
    )

if not is_rdd_split_root(rdd_root):
    raise FileNotFoundError(
        'RDD dataset not found at /content/drive/MyDrive/RDD_SPLIT with expected train/val images+labels structure.'
    )

if not is_pothole_seg_root(pothole_root):
    raise FileNotFoundError(
        'Pothole dataset not found at /content/drive/MyDrive/pothole with expected train/valid images+labels structure.'
    )

print('Using RDD dataset root:', rdd_root)
for p in sorted(rdd_root.iterdir()):
    print(' -', p.name)

print('\nUsing pothole dataset root:', pothole_root)
for p in sorted(pothole_root.iterdir()):
    print(' -', p.name)

FileNotFoundError: Could not find an extracted RDD_SPLIT folder in the cloned repo, Drive, or /content. Place the extracted dataset so it contains train/val/test folders with images and labels.

In [ ]:
# Cell 4: Convert both datasets into a combined ImageFolder layout for classifier training
from pathlib import Path
import shutil

combined_root = Path('/content/HackKU-2026/data/combined_imagefolder')
if combined_root.exists():
    shutil.rmtree(combined_root)
    print('Removed existing combined output:', combined_root)

%cd /content/HackKU-2026
!python prepare_rdd_imagefolder.py --source "{rdd_root}" --output data/combined_imagefolder --splits train val --mode copy --include-empty

pothole_class_name = 'D40_pothole'
valid_image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def copy_pothole_images(src_images_dir: Path, dst_class_dir: Path, prefix: str):
    copied = 0
    dst_class_dir.mkdir(parents=True, exist_ok=True)
    if not src_images_dir.exists():
        return copied

    for image_path in sorted(src_images_dir.iterdir()):
        if not image_path.is_file():
            continue
        if image_path.suffix.lower() not in valid_image_exts:
            continue
        if ':zone.identifier' in image_path.name.lower():
            continue

        dst_name = f"{prefix}_{image_path.name}"
        dst_path = dst_class_dir / dst_name
        if dst_path.exists():
            dst_name = f"{prefix}_{image_path.stem}_{copied}{image_path.suffix}"
            dst_path = dst_class_dir / dst_name

        shutil.copy2(image_path, dst_path)
        copied += 1

    return copied

pothole_train_src = pothole_root / 'train' / 'images'
pothole_valid_src = pothole_root / 'valid' / 'images'

pothole_train_dst = combined_root / 'train' / pothole_class_name
pothole_val_dst = combined_root / 'val' / pothole_class_name

train_added = copy_pothole_images(pothole_train_src, pothole_train_dst, 'pseg_train')
val_added = copy_pothole_images(pothole_valid_src, pothole_val_dst, 'pseg_valid')

print(f'Added pothole segmentation images -> train/{pothole_class_name}: {train_added}')
print(f'Added pothole segmentation images -> val/{pothole_class_name}: {val_added}')

for split in ('train', 'val'):
    split_dir = combined_root / split
    print(f'\n[{split}] class counts:')
    if split_dir.exists():
        for class_dir in sorted([d for d in split_dir.iterdir() if d.is_dir()]):
            count = sum(1 for p in class_dir.iterdir() if p.is_file())
            print(f'  {class_dir.name}: {count}')
    else:
        print('  missing split directory')

In [ ]:
# Cell 5: Train baseline classifier and save artifacts (combined RDD + pothole dataset)
%cd /content/HackKU-2026
!python train_baseline.py --data_dir data/combined_imagefolder/train --epochs 5 --batch_size 32 --output_dir artifacts

In [ ]:
# Cell 6: Verify artifacts and preview validation metrics
import json
from pathlib import Path

artifacts = Path('/content/HackKU-2026/artifacts')
print('Artifacts directory:', artifacts)
!ls -lah /content/HackKU-2026/artifacts

report_path = artifacts / 'val_report.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('\nPer-class precision/recall/f1:')
    for label, stats in report.items():
        if isinstance(stats, dict) and 'f1-score' in stats:
            print(f"{label:30s}  p={stats['precision']:.3f}  r={stats['recall']:.3f}  f1={stats['f1-score']:.3f}")
else:
    print('val_report.json not found yet.')

In [ ]:
# Cell 7: Copy artifacts to Google Drive for reuse
from pathlib import Path

src = Path('/content/HackKU-2026/artifacts')
dst = Path('/content/drive/MyDrive/HackKU_artifacts')
dst.mkdir(parents=True, exist_ok=True)

files = ['baseline_model.pt', 'labels.json', 'val_report.json']
for name in files:
    src_file = src / name
    if src_file.exists():
        !cp -f "{src_file}" "{dst / name}"
    else:
        print('Missing:', src_file)

print('Saved artifacts to:', dst)
!ls -lah /content/drive/MyDrive/HackKU_artifacts

In [ ]:
# Cell 8: Sync this notebook file to Google Drive if it is available in the runtime
from pathlib import Path
import shutil

notebook_name = 'train_dataset.ipynb'
drive_notebook_dir = Path('/content/drive/MyDrive/HackKU_notebooks')
drive_notebook_dir.mkdir(parents=True, exist_ok=True)
target_notebook = drive_notebook_dir / notebook_name

candidate_sources = [
    Path('/content') / notebook_name,
    Path('/content/drive/MyDrive') / notebook_name,
    Path('/content/drive/MyDrive/HackKU_notebooks') / notebook_name,
]

for source in candidate_sources:
    if source.exists() and source.resolve() != target_notebook.resolve():
        shutil.copy2(source, target_notebook)
        print('Synced notebook to:', target_notebook)
        break
else:
    if target_notebook.exists():
        print('Notebook already present in Drive at:', target_notebook)
    else:
        print('Notebook file not available in the runtime to sync automatically.')
        print('Use Drive > Save a copy in Drive if you opened the notebook directly from Colab.')